# 04.7 — Weighted classification loss

Neural decoding data is imbalanced — most trials are correct, few are errors; some feature values appear far more than others ([03.4](../03_data_pipeline/03.4_kfold_stratification_deep_dive.ipynb) fought this at the fold level). A plain cross-entropy loss lets the model win by always guessing the majority class. `WeightedLoss='Inverse'` fixes it by weighting each class inversely to its frequency — rare classes count more. This notebook is about that weighting: what it computes and why it works.

This notebook covers:

* Why unweighted loss fails on imbalanced data.
* Inverse-frequency weights: the formula and the intuition.
* `torch.nn.functional.cross_entropy`'s `weight` argument.
* The per-dimension weights across the multi-head classifier.

**Prerequisites:** [04.6 multi-head classifier](04.6_multi_head_classifier.ipynb), [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb).

## Section 1 — What MATLAB does

`cgg_getWeightsForLoss` computes per-class weights from the training-set label frequencies, and the loss applies them:

```matlab
counts  = histcounts(labels, 1:NumClasses+1);
weights = sum(counts) ./ (NumClasses * counts);   % inverse-frequency
loss    = crossentropy(YPred, T, 'Weights', weights, 'WeightsFormat', 'C');
```

`WeightedLoss = 'Inverse'` selects this; `WeightedLoss = ''` (empty) uses uniform weights — the CC.7 unweighted path. The Python port computes the same weights and hands them to PyTorch's cross-entropy.

## Section 2 — The Python concepts you need

### 2.1 — Why unweighted loss fails

In [ ]:
import torch
import torch.nn.functional as F

# 90% class 0, 10% class 1 — a "predict 0 always" model:
labels = torch.tensor([0]*90 + [1]*10)
always_zero = torch.zeros(100, 2)
always_zero[:, 0] = 10.0                     # huge logit for class 0, always

acc = (always_zero.argmax(1) == labels).float().mean()
loss = F.cross_entropy(always_zero, labels)
print(f"'always predict 0' model:  accuracy {acc:.0%}, unweighted loss {loss:.3f}")
print("90% accuracy and low loss — while being useless on the class you care about.")

The degenerate model scores 90% and a low loss by ignoring the minority class entirely. On error-detection (few errors) or rare feature values, that's exactly the failure mode: high headline numbers, zero useful signal. The optimizer, minimizing unweighted loss, is *rewarded* for this.

### 2.2 — Inverse-frequency weights

Weight each class by the inverse of how often it appears, so a rare class contributes as much total loss as a common one. The production formula (`total / (num_classes × count)`):

In [ ]:
from neural_data_decoding.training.losses.classification import inverse_frequency_class_weights

targets = torch.tensor([[0]]*90 + [[1]]*10)      # (N, num_dims=1): 90 zeros, 10 ones
weights = inverse_frequency_class_weights(targets, num_classes_per_dim=[2])
print("class counts: [90, 10]")
print("inverse-frequency weights:", weights[0].tolist())
print(f"  class 0 (common): {weights[0][0]:.3f}  — down-weighted")
print(f"  class 1 (rare):   {weights[0][1]:.3f}  — up-weighted ~9×")

The rare class gets ~9× the weight of the common one — precisely the imbalance ratio, so each class's *total* contribution to the loss is equalized. The formula `total / (K · count)`:

- class 0: `100 / (2 · 90) = 0.556`
- class 1: `100 / (2 · 10) = 5.000`

A class appearing K× less often gets K× more weight. The intuition: the loss should care about *classes*, not *examples* — inverse weighting converts per-example loss into per-class-balanced loss.

### 2.3 — Applying the weights

In [ ]:
# The SAME degenerate model, now scored with weighted cross-entropy:
w = weights[0]
loss_unweighted = F.cross_entropy(always_zero, labels)
loss_weighted   = F.cross_entropy(always_zero, labels, weight=w)
print(f"unweighted loss: {loss_unweighted:.3f}   (cheap to ignore class 1)")
print(f"weighted loss:   {loss_weighted:.3f}   (class-1 mistakes now cost ~9× more)")
print("The weighted loss PUNISHES ignoring the minority class — so training can't take the shortcut.")

`cross_entropy`'s `weight=` argument takes a per-class tensor and scales each example's loss by its true class's weight. That single argument turns "minimize average error" into "minimize class-balanced error" — the optimizer can no longer buy a low loss by abandoning rare classes.

### 2.4 — Per-dimension weights across the heads

The multi-head classifier ([04.6](04.6_multi_head_classifier.ipynb)) has one head per dimension, each with its *own* class balance — so weights are computed **per dimension**. `inverse_frequency_class_weights` returns a list, one weight-tensor per head:

In [ ]:
# 3 dimensions with different imbalances:
targets = torch.tensor(
    [[0, 1, 2]]*50 + [[1, 1, 0]]*30 + [[0, 0, 1]]*20   # dim0: 70/30, dim1: 80/20, dim2: mixed
)
weights = inverse_frequency_class_weights(targets, num_classes_per_dim=[2, 2, 3])
for d, w in enumerate(weights):
    print(f"  dimension {d}: weights {[round(v, 2) for v in w.tolist()]}")

Each head gets weights tuned to its own dimension's frequencies — the loss for dimension 1 (80/20) is corrected independently of dimension 0 (70/30). This is why the weights are a *list* aligned with the heads, mirroring the list-of-logits output.

**`WeightedLoss=''` (CC.7 unweighted path):** setting the config to empty skips all of this — uniform weights, plain cross-entropy — for the cases where you *want* the model to follow the natural distribution.

## Section 3 — The `neural_data_decoding` implementation

`inverse_frequency_class_weights` — the formula, per dimension:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/classification.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def inverse_frequency_class_weights"))
for k in range(i, min(i + 26, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Counts are tallied per dimension (`torch.bincount` or equivalent), then inverted with the `total / (K · count)` formula from §2.2, citing `cgg_getWeightsForLoss.m`.
* The return is a **list** aligned with `num_classes_per_dim` — one weight tensor per head (§2.4).
* Zero-count classes are handled (a class absent from the training split would divide by zero) — the code guards it; a good example of the edge case a naive port forgets.
* The weights are computed from the **training** labels only — computing them from val/test would leak distribution information across the split ([03.4](../03_data_pipeline/03.4_kfold_stratification_deep_dive.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — verify the formula

For classes with counts `[80, 15, 5]` and the formula `total / (K · count)`, compute the three weights by hand, then check against `inverse_frequency_class_weights`.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
targets = torch.tensor([[0]]*80 + [[1]]*15 + [[2]]*5)
by_hand = [100 / (3 * c) for c in [80, 15, 5]]
actual = inverse_frequency_class_weights(targets, [3])[0].tolist()
print("by hand:", [round(v, 3) for v in by_hand])
print("actual :", [round(v, 3) for v in actual])

### Exercise 4.2 — see the training effect

Take the §2.1 imbalanced setup and show that a class-1 mistake incurs a bigger *weighted* loss than a class-0 mistake of the same confidence — the asymmetry that reshapes the gradient.

In [ ]:
# Reveal:
w = inverse_frequency_class_weights(torch.tensor([[0]]*90 + [[1]]*10), [2])[0]
wrong_on_0 = F.cross_entropy(torch.tensor([[0.0, 3.0]]), torch.tensor([0]), weight=w)  # true 0, predicts 1
wrong_on_1 = F.cross_entropy(torch.tensor([[3.0, 0.0]]), torch.tensor([1]), weight=w)  # true 1, predicts 0
print(f"mistake on common class 0: weighted loss {wrong_on_0:.3f}")
print(f"mistake on rare   class 1: weighted loss {wrong_on_1:.3f}  (~9× — the gradient reflects this)")

## Section 5 — Common errors

### Model hits high accuracy but is useless on the minority class

Unweighted loss on imbalanced data (§2.1) — the majority-class shortcut. Turn on `WeightedLoss='Inverse'`, or report *balanced* accuracy (which the project does — `Scaled_BalancedAccuracy`) so the shortcut can't hide.

### `RuntimeError`: weight tensor size mismatch

`cross_entropy`'s `weight` must have exactly `num_classes` entries for that head. A per-dimension weight tensor applied to the wrong head (different class count) fails here.

### Weights computed from the whole dataset

Compute them from **training** labels only (§3) — val/test frequencies leaking into the weights is a subtle form of information leakage.

### A class absent from training breaks the weights

Zero count → divide by zero → inf/nan weight. The production code guards this; a hand-rolled version must too. It also signals a stratification problem ([03.4](../03_data_pipeline/03.4_kfold_stratification_deep_dive.ipynb)) — a fold missing a class entirely.

### Weighted loss makes training unstable

Extreme imbalance → extreme weights → large, spiky gradients. Gradient clipping ([02.7 §5](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb)) helps; so does capping the weight ratio if one class is vanishingly rare.

## Section 6 — Further reading

- [cross_entropy weight argument](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) — the per-class weighting mechanism.
- [Class imbalance strategies](https://machinelearningmastery.com/tour-of-evaluation-metrics-for-imbalanced-classification/) — weighting, resampling, and the metrics that don't lie.
- [`src/neural_data_decoding/training/losses/classification.py`](../../src/neural_data_decoding/training/losses/classification.py) — the weights + the multi-head cross-entropy that consumes them.

Next up: **[04.8 weight initialization](04.8_weight_initialization_he_vs_pytorch_defaults.ipynb)** — the last notebook of Module 04, on why the codebase sets He init explicitly instead of trusting PyTorch's default.